# Business Complaint Intelligence Platform
## DistilBERT Complaint Category Model - Google Colab

This notebook trains a **DistilBERT multiclass classifier directly on `customer_support_tickets.csv`**.

### Dataset-grounded target

The uploaded dataset contains **8,469 tickets** and 17 columns. Its native complaint/category field is `Ticket Type`, with five classes:

- `Refund request`
- `Technical issue`
- `Cancellation request`
- `Product inquiry`
- `Billing inquiry`

For an honest ML experiment, these **native labels are used as the training target** rather than inventing labels that do not exist in the CSV.

> Your FYP proposal separately describes an AI Complaint Classification Engine with categories such as billing, delivery, product quality, technical issues, and customer service. The current CSV does **not** contain those five labels as ground truth. A later section explains how to handle that mismatch without contaminating your experiment.


## 1. Colab runtime

For training, use:

**Runtime → Change runtime type → GPU → T4 GPU**

The notebook is designed to run on Colab CPU too, but GPU training is much faster.


## 2. Install compatible packages

`pandas` is pinned to `2.2.3` to avoid the Colab package conflict you encountered.


In [ ]:
!pip -q install -U transformers datasets accelerate scikit-learn matplotlib seaborn pandas==2.2.3

## 3. Imports and experiment configuration

In [ ]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

MODEL_NAME = "distilbert/distilbert-base-uncased"
MAX_LENGTH = 256

OUTPUT_DIR = "/content/customer-support-distilbert"

# Reproducible split
TRAIN_SIZE = 0.80
VALIDATION_SIZE = 0.10
TEST_SIZE = 0.10

print("Pandas:", pd.__version__)
print("Transformers:", __import__("transformers").__version__)
print("Datasets:", __import__("datasets").__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 4. Upload `customer_support_tickets.csv`

Run the cell and select your CSV from your computer.

The notebook will then use the uploaded file directly. No Hugging Face dataset conversion is needed.


In [ ]:
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No CSV was uploaded.")

CSV_FILE = next(
    (name for name in uploaded if name.lower().endswith(".csv")),
    next(iter(uploaded))
)

print("Using file:", CSV_FILE)

df = pd.read_csv(CSV_FILE)

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.head())


## 5. Verify the dataset structure

The model needs:

- `Ticket Type` → target label
- `Ticket Subject` → short complaint summary
- `Ticket Description` → complaint details
- `Product Purchased` → used only to replace the `{product_purchased}` placeholder in the text

We do **not** train on customer name, email, age, gender, status, resolution, priority, channel, response time, or satisfaction rating because those fields are metadata or post-complaint information and are not appropriate inputs for a complaint-category model at submission time.


In [ ]:
REQUIRED_COLUMNS = [
    "Ticket Type",
    "Ticket Subject",
    "Ticket Description",
    "Product Purchased",
]

missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Required columns found.")


## 6. Inspect the native target labels

These are the real labels contained in this CSV.


In [ ]:
label_counts = df["Ticket Type"].value_counts(dropna=False)

print("Number of classes:", df["Ticket Type"].nunique(dropna=True))
display(label_counts.to_frame("count"))

plt.figure(figsize=(9, 4))
sns.countplot(
    data=df,
    x="Ticket Type",
    order=label_counts.index
)
plt.xticks(rotation=25, ha="right")
plt.title("Native Ticket Type Distribution")
plt.xlabel("Ticket Type")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## 7. Build the model input text

For complaint classification, the model receives the complaint language itself.

We combine:

```text
Ticket Subject + Ticket Description
```

We also replace the dataset placeholder `{product_purchased}` with the real product name where available.

This keeps the inference path realistic: a new complaint can be passed to the model as text without relying on private customer metadata.


In [ ]:
work = df.copy()

for col in ["Ticket Subject", "Ticket Description", "Product Purchased", "Ticket Type"]:
    work[col] = work[col].fillna("").astype(str).str.strip()

# Replace literal template placeholders with the actual product name.
work["Ticket Description"] = work.apply(
    lambda r: r["Ticket Description"].replace(
        "{product_purchased}",
        r["Product Purchased"] if r["Product Purchased"] else "product"
    ),
    axis=1,
)

work["text"] = (
    "Subject: " + work["Ticket Subject"] +
    "\nDescription: " + work["Ticket Description"]
)

work = work.rename(columns={"Ticket Type": "label"})
work = work[["text", "label"]]

# Basic cleaning
work["text"] = (
    work["text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
work["label"] = work["label"].str.strip()

work = work[work["text"].str.len() >= 10]
work = work[work["label"] != ""]

print("Rows before exact duplicate removal:", len(work))

# Remove exact duplicate text-label pairs before splitting.
work = work.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

print("Rows after exact duplicate removal:", len(work))
display(work.head())


## 8. Final class list and label IDs

The class IDs are fixed from the dataset, so the saved model can be integrated consistently later.


In [ ]:
LABELS = sorted(work["label"].unique().tolist())

LABEL2ID = {label: idx for idx, label in enumerate(LABELS)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}

print("Labels:")
for k, v in LABEL2ID.items():
    print(f"{v}: {k}")

assert len(LABELS) == 5, (
    f"Expected 5 native Ticket Type classes, found {len(LABELS)}: {LABELS}"
)


## 9. Stratified train / validation / test split

We use:

- 80% training
- 10% validation
- 10% held-out test

Stratification keeps all five ticket types represented proportionally.


In [ ]:
train_df, temp_df = train_test_split(
    work,
    test_size=(VALIDATION_SIZE + TEST_SIZE),
    random_state=SEED,
    stratify=work["label"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=TEST_SIZE / (VALIDATION_SIZE + TEST_SIZE),
    random_state=SEED,
    stratify=temp_df["label"],
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTrain distribution:")
display(train_df["label"].value_counts().reindex(LABELS).to_frame("count"))

print("\nValidation distribution:")
display(val_df["label"].value_counts().reindex(LABELS).to_frame("count"))

print("\nTest distribution:")
display(test_df["label"].value_counts().reindex(LABELS).to_frame("count"))


## 10. Convert to Hugging Face datasets

In [ ]:
def to_hf_dataset(frame):
    x = frame.copy()
    x["labels"] = x["label"].map(LABEL2ID).astype(int)
    return Dataset.from_pandas(
        x[["text", "labels"]],
        preserve_index=False
    )

dataset = DatasetDict({
    "train": to_hf_dataset(train_df),
    "validation": to_hf_dataset(val_df),
    "test": to_hf_dataset(test_df),
})

dataset


## 11. Load DistilBERT tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized = dataset.map(
    tokenize_batch,
    batched=True,
    desc="Tokenizing"
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print(tokenized)


## 12. Load DistilBERT sequence-classification model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

print("Base model:", MODEL_NAME)
print("Number of labels:", len(LABELS))


## 13. Evaluation metrics

We report accuracy plus macro and weighted Precision, Recall and F1.

**Macro F1** is the main balanced metric for the FYP because it gives every class equal weight.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    precision_weighted, recall_weighted, f1_weighted, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted",
            zero_division=0,
        )
    )

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_precision": precision_macro,
        "macro_recall": recall_macro,
        "macro_f1": f1_macro,
        "weighted_precision": precision_weighted,
        "weighted_recall": recall_weighted,
        "weighted_f1": f1_weighted,
    }


## 14. Training configuration

These settings are chosen to be practical on a Colab T4 GPU.

The model uses:

- learning rate: `2e-5`
- batch size: `16`
- gradient accumulation: `2`
- effective batch size: about `32`
- up to 4 epochs
- early stopping based on validation macro F1
- best checkpoint restored automatically


In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,

    num_train_epochs=4,
    weight_decay=0.01,
    warmup_ratio=0.10,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    logging_steps=25,
    save_total_limit=2,

    fp16=torch.cuda.is_available(),
    report_to="none",

    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)
    ],
)


## 15. Train DistilBERT

In [ ]:
train_result = trainer.train()

print("Training complete.")
print(train_result)


## 16. Evaluate on the held-out test set

These results are the numbers you should use for the final experiment, not the training accuracy.


In [ ]:
test_result = trainer.predict(tokenized["test"])

y_true = test_result.label_ids
y_pred = np.argmax(test_result.predictions, axis=1)

test_metrics = compute_metrics(
    (test_result.predictions, y_true)
)

print("Test metrics:")
print(json.dumps(test_metrics, indent=2))

print("\nPer-class classification report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(len(LABELS))),
        target_names=LABELS,
        digits=4,
        zero_division=0,
    )
)


## 17. Confusion matrix

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(LABELS)))
)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=LABELS,
    yticklabels=LABELS,
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("DistilBERT - Ticket Type Confusion Matrix")
plt.xticks(rotation=25, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 18. Inspect misclassified tickets

This is important for the FYP because it tells you which categories the model confuses.


In [ ]:
error_df = test_df.copy()
error_df["actual"] = [ID2LABEL[int(x)] for x in y_true]
error_df["predicted"] = [ID2LABEL[int(x)] for x in y_pred]
error_df["correct"] = error_df["actual"] == error_df["predicted"]

misclassified = error_df[~error_df["correct"]].copy()

print("Misclassified samples:", len(misclassified))
display(
    misclassified[
        ["text", "actual", "predicted"]
    ].head(20)
)


## 19. Save the trained model and tokenizer

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(os.path.join(OUTPUT_DIR, "label_mapping.json"), "w") as f:
    json.dump(
        {
            "label2id": LABEL2ID,
            "id2label": {str(k): v for k, v in ID2LABEL.items()},
            "model_name": MODEL_NAME,
            "max_length": MAX_LENGTH,
        },
        f,
        indent=2,
    )

pd.DataFrame([test_metrics]).to_csv(
    os.path.join(OUTPUT_DIR, "test_metrics.csv"),
    index=False
)

print("Model saved to:", OUTPUT_DIR)


## 20. Live complaint prediction

Type a new complaint. The model returns the most likely native `Ticket Type` plus confidence scores.


In [ ]:
def predict_ticket(text, top_k=5):
    model = trainer.model
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    device = model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0]
        probabilities = probabilities.detach().cpu().numpy()

    top_k = min(top_k, len(LABELS))
    order = np.argsort(probabilities)[::-1][:top_k]

    result = pd.DataFrame({
        "ticket_type": [ID2LABEL[int(i)] for i in order],
        "confidence": [float(probabilities[i]) for i in order],
    })

    return result


complaint = input("Enter a complaint: ")

display(
    predict_ticket(
        "Subject: Complaint\nDescription: " + complaint
    )
)


## 21. Example test complaints

In [ ]:
examples = [
    "I want my money back because the item did not meet my expectations.",
    "The device is not working after I installed the software.",
    "I need to cancel my order before it is processed.",
    "Can you tell me which product is best for my needs?",
    "I was charged incorrectly for my purchase.",
]

for text in examples:
    print("=" * 90)
    print("Complaint:", text)
    display(
        predict_ticket(
            "Subject: Complaint\nDescription: " + text,
            top_k=3
        )
    )


## 22. Download the trained model

The ZIP can be uploaded to your backend later for inference.


In [ ]:
import shutil

zip_base = "/content/customer-support-distilbert"

shutil.make_archive(
    zip_base,
    "zip",
    OUTPUT_DIR
)

MODEL_ZIP = zip_base + ".zip"

print("Created:", MODEL_ZIP)
files.download(MODEL_ZIP)


# FYP category mismatch: what to do

Your FYP proposal defines example business complaint categories including:

- billing
- delivery
- product quality
- technical issues
- customer service

The current CSV does **not** label tickets with those five categories. Its ground-truth `Ticket Type` field uses:

- Refund request
- Technical issue
- Cancellation request
- Product inquiry
- Billing inquiry

So you should **not** silently claim that this training run is a classifier for the FYP's five proposed categories.

### Recommended research approach

**Experiment A - Dataset benchmark**

Train the notebook exactly as written, using the native `Ticket Type` labels. This gives you a clean, reproducible benchmark for the dataset you actually have.

**Experiment B - Final FYP model**

Create a new manually labelled dataset with your FYP's five target categories:

`billing`, `delivery`, `product_quality`, `technical`, `customer_service`

Then fine-tune the same DistilBERT pipeline on those labels.

That second experiment is the one that should power your FYP's complaint-routing feature.
